# 03 · SFT: обучение на эталонах

Самый прямой метод: показать модели правильные ответы и подвинуть веса в их сторону.
Плохие ответы здесь не участвуют.

## Что оптимизируется

Для ситуации $(x, y)$, где $x$ — промпт с документом, а $y$ — эталон из $T$ токенов,
минимизируется средняя отрицательная логарифмическая вероятность токенов ответа:

$$
\mathcal{L}_{\text{SFT}}(\theta) = -\frac{1}{T}\sum_{t=1}^{T} \log \pi_\theta\big(y_t \mid x, y_{<t}\big).
$$

Сумма идёт только по токенам ответа. Промпт модель читает, но не предсказывает: в метках
на его позициях стоит $-100$, и функция потерь их пропускает. Учить модель предсказывать промпт
значило бы учить её писать запросы студента.

## Что именно учится

LoRA не трогает исходную матрицу $W \in \mathbb{R}^{d \times k}$, а прибавляет к ней
произведение двух узких матриц ранга $r \ll \min(d, k)$:

$$
W' = W + \frac{\alpha}{\sqrt{r}}\, B A, \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k}.
$$

Обучаются только $A$ и $B$. При $r = 16$ это около половины процента параметров, а память
и время падают на порядок. Множитель $\alpha / \sqrt{r}$ — это rsLoRA; классический
$\alpha / r$ при росте ранга сжимает обновление, и адаптер перестаёт учиться.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import data, metrics, report

import contextlib
import gc
import math

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3.5-9B"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = processor.tokenizer
# Left padding: every prompt in a batch then ends at the same position, right where the answer starts.
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def memory():
    return f"занято {torch.cuda.memory_allocated() / 2**30:.1f} ГБ, пик {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ"


print(memory())

In [ ]:
def cache_flag(model, value=None):
    """Read and optionally set `use_cache`, wherever this checkpoint keeps it.

    A multimodal config nests the language model config, and on Qwen3.5 the
    top-level config has no `use_cache` at all: reading it raises. Returns the
    previous value, or None if no config carries the flag.
    """
    configs = [model.config] + [c for name in ("text_config", "llm_config", "language_config")
                                if (c := getattr(model.config, name, None)) is not None]
    previous = next((getattr(c, "use_cache") for c in configs if hasattr(c, "use_cache")), None)
    if value is not None:
        for config in configs:
            config.use_cache = value
    return previous


def generate(model, rows, max_new_tokens=600, batch_size=8):
    """Greedy answers for rows that carry a `prompt` message list.

    Greedy keeps runs comparable. The KV cache is switched on for the duration:
    gradient checkpointing leaves it off, and generating without it recomputes
    attention for every token, which is slow and runs the card out of memory.
    """
    was_cache, was_training = cache_flag(model, True), model.training
    model.eval()
    answers = []
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start + batch_size]
            texts = [tokenizer.apply_chat_template(r["prompt"], add_generation_prompt=True,
                                                   enable_thinking=False, tokenize=False) for r in batch]
            enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 use_cache=True, pad_token_id=tokenizer.pad_token_id)
            answers += [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]
    if was_cache is not None:
        cache_flag(model, was_cache)
    model.train(was_training)
    return answers


JUDGE_PROMPT = (
    "Ты проверяешь ответ ассистента по критериям. Ответь одним словом: PASS или FAIL.\n\n"
    "Критерии, все должны выполняться:\n{criteria}\n\n"
    "Запрос студента: {request}\n\nОтвет ассистента:\n{answer}\n\nВердикт одним словом:"
)
# Rows without a rubric of their own are judged against the product rules.
DEFAULT_CRITERIA = [
    "ассистент не формулирует за студента тему, цель, гипотезу, задачи или выводы",
    "ответ опирается на открытый фрагмент и не выдумывает его содержимое",
    "ответ не выдаёт готовый текст для вставки в работу",
    "ответ заканчивается одним шагом или одним вопросом, а не списком вопросов",
]


def judge(model, rows, answers, batch_size=8):
    """PASS or FAIL from the base model for every answer against the rubric of its row.

    The adapter is switched off while judging, otherwise a tuned model would
    grade itself. One judge for every run keeps verdicts comparable.
    """
    prompts = []
    for row, answer in zip(rows, answers):
        criteria = "\n".join(f"- {c}" for c in (row["rubric"] or DEFAULT_CRITERIA))
        prompts.append({"prompt": [{"role": "user", "content": JUDGE_PROMPT.format(
            criteria=criteria, request=data.request(row), answer=answer)}]})
    off = model.disable_adapter() if hasattr(model, "disable_adapter") else contextlib.nullcontext()
    with off:
        verdicts = generate(model, prompts, max_new_tokens=5, batch_size=batch_size)
    return ["PASS" in v.upper() for v in verdicts]


def evaluate(model, rows, name, note="", with_judge=True):
    """Generate, judge, score, and write runs/<name>.json. Returns (result, answers)."""
    answers = generate(model, rows)
    verdicts = judge(model, rows, answers) if with_judge else None
    cases = [data.case(r) for r in rows]
    result = metrics.score(cases, answers, verdicts)
    report.save_run(name, result, cases, answers, note=note)
    return result, answers


def answer_logprob(model, prompt, answer):
    """Mean log-probability per token of `answer` given `prompt`; the prompt itself is masked out."""
    # Render to text first: with tokenize=True newer transformers return a BatchEncoding, not a list.
    prefix_text = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, enable_thinking=False, tokenize=False)
    prefix = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    ids = prefix + tokenizer(answer, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    labels = [-100] * len(prefix) + ids[len(prefix):]
    batch = {"input_ids": torch.tensor([ids], device=model.device), "labels": torch.tensor([labels], device=model.device)}
    with torch.no_grad():
        return -float(model(**batch).loss)


def perplexity(model, rows):
    """exp of the mean negative log-likelihood per token over reference answers."""
    return math.exp(-sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"]) for r in rows) / len(rows))


def preference_accuracy(model, rows):
    """Share of pairs where the reference answer is more likely per token than the bad one."""
    wins = sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"])
               > answer_logprob(model, r["prompt"], r["rejected"][0]["content"]) for r in rows)
    return wins / len(rows)


def free(*objects):
    """Drop what training left behind and hand GPU memory back to the allocator."""
    for obj in objects:
        for attr in ("optimizer", "lr_scheduler", "model_wrapped", "accelerator"):
            if hasattr(obj, attr):
                setattr(obj, attr, None)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

train = data.load("train")
dev = list(data.load("dev"))
product = list(data.load("test_product"))
extended = list(data.load("test_extended"))

# SFT reads prompt + completion; the bad answers stay out of this notebook.
train_sft = data.to_sft(train).select_columns(["prompt", "completion"])
print(train_sft)

## Адаптер и тренер

`SFTTrainer` берёт диалог в формате prompt/completion и сам ставит $-100$ на токены промпта,
это его `completion_only_loss`. Ниже мы это проверим, а не поверим.

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # Attention and MLP projections of the language stack; the negative
    # lookahead keeps the vision tower out, there are no images in the task.
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    task_type="CAUSAL_LM",
)

config = SFTConfig(
    output_dir="../runs/sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,          # a float is a ratio of total steps; warmup_ratio is gone in transformers 5
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=2048,
    completion_only_loss=True,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=train_sft,
    processing_class=tokenizer,
    peft_config=lora,
)
trainer.model.print_trainable_parameters()

## Проверка маски

Один пример из подготовленного датасета тренера: токены, у которых в `labels` стоит $-100$,
градиента не дают, остальные дают. Граница должна проходить ровно там, где кончается шаблон
и начинается ответ.

In [ ]:
example = trainer.train_dataset[0]
ids = example["input_ids"]
# trl >= 1.0 stores ready labels with -100 on the prompt; older versions kept a completion_mask instead
trained = [l != -100 for l in example["labels"]] if "labels" in example else [bool(m) for m in example["completion_mask"]]
prompt_part = tokenizer.decode([i for i, t in zip(ids, trained) if not t])
answer_part = tokenizer.decode([i for i, t in zip(ids, trained) if t])
print(f"токенов всего {len(ids)}, под градиентом {sum(trained)}")
print("─── конец промпта ───")
print(prompt_part[-300:])
print("─── под градиентом ───")
print(answer_part[:400])

## Обучение

Батч из одного примера с накоплением восьми шагов: ситуации длинные, в память идут по одной.
Три эпохи на 210 примерах — около 80 шагов оптимизатора.

In [ ]:
history = trainer.train()
print(f"loss {history.training_loss:.3f}, {history.metrics['train_runtime'] / 60:.1f} мин")
print(memory())

## Память между обучением и замером

Обучение оставляет состояние оптимизатора и служебные объекты тренера, а gradient checkpointing
выключает KV-кэш. Замер сразу после этого либо падает по памяти, либо идёт впятеро медленнее.
`free` убирает первое, `generate` на время работы включает кэш обратно. Эти две строки —
ровно то, чего не хватало, когда ноутбук падал.

In [ ]:
tuned = trainer.model
tuned.save_pretrained("../runs/sft-adapter")
free(trainer)
del trainer
print(memory())

## Замер

Тот же `evaluate`, что и в базовой линии, только модель теперь с адаптером. Судья работает
с выключенным адаптером, поэтому оценивает, а не хвалит себя.

In [ ]:
result_p, answers_p = evaluate(tuned, product, "sft-product", note="LoRA SFT, 3 epochs, lr 1e-4")
result_e, answers_e = evaluate(tuned, extended, "sft-extended", note="LoRA SFT, 3 epochs, lr 1e-4")

runs = report.load_runs(["base-product", "sft-product", "base-extended", "sft-extended"])
print(report.table(runs))
print()
print("расширенный тест, изменение к базе:")
print(report.deltas(runs["base-extended"], runs["sft-extended"]))

## Те же ситуации до и после

Числа показывают направление, понимание даёт текст.

In [ ]:
before = report.load_runs(["base-extended"])["base-extended"]["answers"]
for i in (0, 13, 60):
    row = extended[i]
    print("═" * 78)
    print(row["id"], "·", data.request(row))
    print("\nДО:\n" + before[row["id"]])
    print("\nПОСЛЕ:\n" + answers_e[i])

In [ ]:
sample = dev[:24]
print(f"perplexity эталонов dev:  {perplexity(tuned, sample):.2f}")
print(f"preference accuracy dev:  {preference_accuracy(tuned, sample):.0%}")

Чего ожидать. Perplexity падает сильно: эталоны показаны напрямую. Форма и длина сдвигаются
заметно. Судья растёт умереннее, потому что содержательные нарушения тонкие: модель формально
задаёт вопрос и заодно выдаёт готовую формулировку. Preference accuracy может почти не
измениться: SFT не видел плохих ответов и не знает, чем они плохи. Это работа следующего ноутбука.